In [1]:
!pip install langchain langchain-classic langchain-community langchain-google-genai langchain-text-splitters sentence-transformers faiss-cpu python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed

In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore

from sentence_transformers import SentenceTransformer
from langchain_core.embeddings import Embeddings

from google.colab import userdata

# Get the API key from Colab's user data
gemini_api_key = userdata.get('GeminiAPIKey1')

class LocalEmbeddings(Embeddings):
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()

    def embed_query(self, text):
        return self.model.encode([text])[0].tolist()

#load_dotenv()

def load_documents(policy_dir):
    policy_dir = Path(policy_dir)
    documents = []
    for file_path in policy_dir.glob("*.txt"):
        with open(file_path, "r", encoding="utf-8") as file:
            content = file.read()
        metadata = {'source_file': file_path.name, 'policy_type': file_path.stem.replace("_policy", "").upper() }
        documents.append(Document(page_content=content, metadata=metadata))

    return documents

def build_parent_child_retriever(documents):
    child_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=0)
    parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1000)

    #embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")


    embeddings = LocalEmbeddings()

    vectorstore = FAISS.from_documents(
        [Document(page_content="init", metadata={})],
        embeddings
    )

    docstore = InMemoryStore()
    retriever = ParentDocumentRetriever(
        vectorstore=vectorstore,
        child_splitter=child_splitter,
        parent_splitter=parent_splitter,
        docstore=docstore,
    )

    retriever.add_documents(documents)
    return retriever

def query_rag(retriever, llm, query):
    parent_docs = retriever.invoke(query)

    context = "\n\n--\n".join([f"[{i}] {doc.metadata.get('policy_type', 'Unknown Policy')} | {doc.metadata.get('source_file', 'Unknown Source')} | {doc.page_content}" for i, doc in enumerate(parent_docs, 1)])

    prompt = f"""
    You are a helpful assistant.
    Use the following information to answer the question.
    {context}
    Question: {query}
    INSTRUCTIONS:
    1. Answer the question based ONLY on the provided context
    2. Be specific and cite which policy section you're referencing
    3. If the context doesn't contain the answer, say "I don't have that information in the policies provided"
    4. Keep answer concise but complete (1-2 paragraphs max)
    5. Include relevant details like timeframes, contact info, or process steps
    """

    response = llm.invoke(prompt)

    return response.content, parent_docs



if __name__ == "__main__":
    policy_dir = "/content/"
    docs = load_documents(policy_dir)
    retriever = build_parent_child_retriever(docs)

    llm = ChatGoogleGenerativeAI(
        model = "gemini-2.5-flash",
        temperature = 0.3,
        api_key = gemini_api_key  # Pass the API key here
    )

    while True:
        query = input("Ask: ")
        if query.lower() in ["exit", "quit"]:
            break

        answer, parent_docs = query_rag(retriever, llm, query)
        print(answer)
        print("Sources: ")
        for doc in parent_docs:
            print(f"  - {doc.metadata['policy_type']} | {doc.page_content[:100]}...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ask: What happens if a PO number is missing?
If a PO number is missing from an invoice, it will hinder automated processing and matching, as the PO number is a mandatory requirement for invoice submission (P2P | p2p_policy.txt | 5. INVOICE PROCESSING). While the policy doesn't explicitly state the direct consequence of a missing PO number, it implies that it could lead to payment delays, similar to how mismatches in line item details can cause such delays.
Sources: 
  - P2P | 5. INVOICE PROCESSING
Vendors submit invoices to: ap@company.com

Required information:
- PO number ...
  - P2P | Typical onboarding time: 5-7 business days.

3. PURCHASE PO CREATION
POs are automatically generated...
  - P2P | 7. DISPUTE RESOLUTION
For invoice disputes:
- Contact procurement team: procurement@company.com
  Se...
Ask: Explain the full invoice submission process
Vendors submit invoices to ap@company.com ([2] P2P | p2p_policy.txt, [3] P2P | p2p_policy.txt). These invoices must include a mandatory 10